# Notebook 04: EXPLAIN Plans & Indexing

**Phase 3 — EXPLAIN Plans & Indexing (PostgreSQL)**

> This is the most differentiating section in the portfolio. Almost no
> portfolio projects demonstrate query plan analysis.

`EXPLAIN` shows the execution plan the planner chose — the sequence of
operations PostgreSQL will perform and their estimated costs.
`EXPLAIN ANALYZE` actually runs the query and adds real timing and row counts,
making it possible to compare what the planner *expected* against what
*actually happened*.

| Section | Topic |
|:---|:---|
| 1 | Reading an EXPLAIN plan — nodes, cost, rows |
| 2 | Seq Scan → Index Scan on `orders` and `lineitem` |
| 3 | Stale statistics and `ANALYZE` |
| 4 | Join strategies — Nested Loop, Hash Join, Merge Join |

---

## Prerequisites

Notebook 01 must have been run first. Indexes are created in this notebook
and are safe to re-run (all use `IF NOT EXISTS`).


In [ ]:
import pathlib
import sys
import time

import psycopg2
import pandas as pd

ROOT = pathlib.Path.cwd()
while not (ROOT / 'pyproject.toml').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from config import settings

pg = psycopg2.connect(settings.dsn)
pg.autocommit = True  # DDL (CREATE INDEX) must not run inside a transaction block
print(f"PostgreSQL connected: {settings.POSTGRES_HOST}:{settings.POSTGRES_PORT}/{settings.POSTGRES_DB}")


SQL_DIR = ROOT / "sql" / "explain"


def load_section(filename: str, section: str) -> str:
    text = (SQL_DIR / filename).read_text(encoding="utf-8")
    blocks: dict[str, str] = {}
    current: str | None = None
    acc: list[str] = []
    for line in text.splitlines():
        if line.startswith("-- §"):
            if current is not None:
                blocks[current] = "\n".join(acc).strip()
            current = line[4:].strip()
            acc = []
        else:
            acc.append(line)
    if current is not None:
        blocks[current] = "\n".join(acc).strip()
    if section not in blocks:
        raise KeyError(f"Section '{section}' not found in {filename}. Available: {list(blocks)}")
    return blocks[section]


def explain(sql: str) -> None:
    with pg.cursor() as cur:
        cur.execute(sql)
        for row in cur.fetchall():
            print(row[0])


def run_timed(sql: str, label: str) -> float:
    with pg.cursor() as cur:
        t0 = time.perf_counter()
        cur.execute(sql)
        cur.fetchall()
        elapsed = time.perf_counter() - t0
    print(f"{label:40s}  {elapsed * 1000:.1f} ms")
    return elapsed


---

## 1 · Reading an EXPLAIN Plan

A plan is a tree of **nodes**.  Each node is one operation; nodes are indented
to show their parent-child relationship.  Data flows *up* the tree — a child
node feeds rows to its parent.

```
Aggregate  (cost=120541.00..120541.01 rows=1 width=8)
  ->  Hash Join  (cost=4985.00..113238.50 rows=2921000 width=4)
        Hash Cond: (l.l_orderkey = o.o_orderkey)
        ->  Seq Scan on lineitem  (cost=0.00..97222.44 rows=5916544 width=8)
        ->  Hash  (cost=4113.00..4113.00 rows=69760 width=4)
              ->  Seq Scan on orders  (cost=0.00..4113.00 rows=69760 width=4)
                    Filter: (o_orderdate > '1995-01-01'::date)
```

**Anatomy of a cost annotation:** `cost=0.43..1427.57 rows=1000 width=8`

| Field | Meaning |
|:---|:---|
| First cost number | **Startup cost** — abstract units paid before the first row is returned (e.g. building a hash table, sorting) |
| Second cost number | **Total cost** — abstract units to return all rows |
| `rows` | Planner's estimate of output row count (from table statistics) |
| `width` | Estimated average row width in bytes |

Cost units are dimensionless — one unit ≈ reading one 8 KB page sequentially.
CPU operations are scaled via `cpu_tuple_cost` etc. in `postgresql.conf`.
They are useful for *comparing* plans, not predicting wall-clock time.

**The gap that matters:** when `EXPLAIN ANALYZE` shows `rows=6000000` (estimate)
but `actual rows=160000`, the planner chose a plan based on wrong assumptions.
The fix: `ANALYZE table_name` to refresh statistics.


---

## 2 · Sequential Scan vs Index Scan

**Sequential Scan** reads every page of the table in order.  Cost is
proportional to table size.  Always chosen when the query touches a large
fraction of the rows — the overhead of an index lookup per row would be worse.

**Index Scan** uses a B-tree to jump directly to matching rows.  Efficient
when the filter is highly selective (few matching rows).  Has a small startup
cost for the index traversal.

**Bitmap Index Scan + Bitmap Heap Scan** is a hybrid: build a bitmap of
matching page locations via the index, sort them into disk order, then fetch
heap pages.  Chosen when the result set is moderate — enough rows that random
Index Scan I/O would be expensive, but not so many that a Seq Scan is better.


In [ ]:
# Seq Scan baseline — orders filtered by a single date, no index yet.
print("=== orders: no index ===")
explain(load_section("01_seq_vs_index_scan.sql", "explain_seq_scan_orders"))


In [ ]:
# Create indexes on the date columns.  IF NOT EXISTS makes this safe to re-run.
print(load_section("01_seq_vs_index_scan.sql", "create_index_orders"))
with pg.cursor() as cur:
    cur.execute(load_section("01_seq_vs_index_scan.sql", "create_index_orders"))
print("idx_orders_orderdate  created")

print(load_section("01_seq_vs_index_scan.sql", "create_index_lineitem"))
with pg.cursor() as cur:
    cur.execute(load_section("01_seq_vs_index_scan.sql", "create_index_lineitem"))
print("idx_lineitem_shipdate created")


In [ ]:
# Same query after index creation — compare the plan node and cost numbers.
print("=== orders: with index ===")
explain(load_section("01_seq_vs_index_scan.sql", "explain_index_scan_orders"))


In [ ]:
# Seq Scan on lineitem (6 M rows) — cost is much more visible at this scale.
print("=== lineitem: no index ===")
explain(load_section("01_seq_vs_index_scan.sql", "explain_seq_scan_lineitem"))


In [ ]:
# lineitem after index — note whether PostgreSQL chooses Index Scan or Bitmap.
# For l_shipdate = '1998-09-01' (~2.6% of rows), Bitmap is common.
print("=== lineitem: with index ===")
explain(load_section("01_seq_vs_index_scan.sql", "explain_index_scan_lineitem"))


In [ ]:
# Wall-clock comparison: same filter query timed before/after index.
# Re-run once without caching effect by running the Seq Scan first via a
# different date, then the index path with the target date.
filter_sql_seq  = "SELECT o_orderkey, o_totalprice FROM orders WHERE o_orderdate = DATE '1994-01-15'"
filter_sql_idx  = "SELECT o_orderkey, o_totalprice FROM orders WHERE o_orderdate = DATE '1995-01-15'"

# Warm the buffer cache with a full scan pass (approximates no-index baseline)
with pg.cursor() as cur:
    cur.execute("SELECT COUNT(*) FROM orders")
    cur.fetchall()

t_seq = run_timed(filter_sql_seq, "approx. Seq Scan path (first run)")
t_idx = run_timed(filter_sql_idx, "Index Scan path")
print(f"\nSpeedup: {t_seq / t_idx:.1f}x")


---

## 3 · Stale Statistics and ANALYZE

PostgreSQL's planner uses **statistics** (column histograms, distinct value
counts, null fractions) sampled with `ANALYZE` to estimate `rows` in each
plan node.

A large gap between estimated and actual rows means the planner is working
with stale or misleading statistics — it may choose a suboptimal join order,
use a hash join where a nested loop would be faster, or skip an index.

Two causes:
- **Stale statistics** — rows inserted/updated since last `ANALYZE`
- **Skewed data** — the histogram bins assume roughly uniform distribution;
  a column where 95% of rows share one value will confuse the planner

`ANALYZE table_name` resamples the table and rebuilds all histograms.
`VACUUM ANALYZE` does both maintenance operations in one pass.
`autovacuum` runs this automatically in production, but after a bulk load it
has not yet had a chance to run.


In [ ]:
# Refresh statistics on the two largest tables after the bulk load.
print(load_section("01_seq_vs_index_scan.sql", "analyze_tables"))
with pg.cursor() as cur:
    cur.execute("ANALYZE orders")
    cur.execute("ANALYZE lineitem")
print("Statistics refreshed.")

# Confirm: run EXPLAIN again — estimated rows should be closer to actual.
print("\n=== orders after ANALYZE ===")
explain(load_section("01_seq_vs_index_scan.sql", "explain_index_scan_orders"))


---

## 4 · Join Strategies

PostgreSQL has three join algorithms.  The planner estimates the cost of each
and picks the cheapest — but understanding which conditions favour which
strategy makes it possible to diagnose a wrong choice.

| Algorithm | How it works | Best when |
|:---|:---|:---|
| **Nested Loop** | For each outer row, probe inner for matches | Outer is small, inner has an index |
| **Hash Join** | Build hash table from smaller input; probe with larger | Both inputs large, no useful index |
| **Merge Join** | Advance two sorted pointers simultaneously | Both inputs pre-sorted on the join key |


In [ ]:
# Nested Loop: nation (25 rows) joined to customer (150 K).
# Outer is tiny; planner does 25 index lookups into customer.
print("=== Nested Loop: nation x customer ===")
explain(load_section("02_join_strategies.sql", "nested_loop_join"))


In [ ]:
# Hash Join: customer (150 K) x orders (1.5 M), both large, no index on o_custkey.
# Planner builds hash table from customer (smaller), probes with orders.
print("=== Hash Join: customer x orders ===")
explain(load_section("02_join_strategies.sql", "hash_join"))


In [ ]:
# Merge Join opportunity: orders x lineitem on o_orderkey (PK/FK).
# Both PKs are indexed; PostgreSQL may use an index to provide sorted order
# and choose Merge Join — or Hash Join if it estimates that cheaper.
# With a date-range filter reducing orders significantly, nested loop is also feasible.
print("=== orders x lineitem (date-filtered) ===")
explain(load_section("02_join_strategies.sql", "merge_join"))


---

## Summary

| Concept | Key point |
|:---|:---|
| `cost=X..Y` | Abstract cost units — X is startup, Y is total; not wall-clock time |
| `rows` estimate vs actual | Large gap = stale statistics; fix with `ANALYZE` |
| Seq Scan | Always reads full table; chosen when filter selectivity is low |
| Index Scan | O(log N) lookup; chosen for highly selective filters |
| Bitmap Scan | Hybrid; chosen for moderate result sets to reduce random I/O |
| Nested Loop | Small outer, indexed inner — low startup cost |
| Hash Join | Large unindexed inputs — high startup (build phase), O(N+M) |
| Merge Join | Pre-sorted inputs — no hash overhead, O(N+M) comparisons |
| `ANALYZE` | Refreshes column histograms; fixes row estimate drift |
| `BUFFERS` option | Shows shared/local buffer hits vs disk reads in `EXPLAIN ANALYZE` |

**Next:** [Notebook 05 — DuckDB Feature Parity](05_duckdb_vs_postgres.ipynb)
